# Corporate Capture and Lost Trust: ANES 2024

Notebook 14 found the 2024 leavers were price voters who had come to believe Republicans would handle prices better, and that what separated them from Democrats who stayed was cultural conservatism and anti-elite attitude, not the priority given immigration. This notebook tests a sharper hypothesis: that the leavers saw the Democratic Party as beholden to corporate and moneyed interests, that this cost the party their trust on the economy, and that the "moderate middle" the party courts is not where the votes went.

Three things are separable in the data:

1. **Anti-corporate vs anti-politician.** The ANES asks a big-business thermometer, a labor-union thermometer, "because of the rich and powerful it's difficult for the rest of us to get ahead," a millionaire tax, and campaign-spending limits — alongside the anti-politician items (corrupt, don't care, strong leader). If the leavers are anti-corporate, the capture story has a base; if they are anti-politician but neutral on big business, it does not.
2. **What they thought of the Democratic Party specifically.** Party thermometers (pre-election), CSES like/dislike, and left-right placement of each party and self. Did leavers see the Democrats as too far left, or as not different enough from Republicans?
3. **Who was reachable.** Self-placement and issue positions of the leavers against the "moderate" the party courts.

**Inputs:** `data/raw/anes/anes_2024_core.csv`. Same groups, weight, and N caveats as notebook 14 (69 switchers).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables')
d = pd.read_csv(RAW_DIR / 'anes' / 'anes_2024_core.csv')
d = d[d['weight_post'].notna() & (d['weight_post'] > 0)].copy()
d['wt'] = d['weight_post']
def wmean(x, w):
    m = x.notna() & w.notna()
    return np.average(x[m], weights=w[m]) if m.any() else np.nan
def wshare(mask, w): return w[mask].sum() / w.sum()
def grp_table(df, cols, order):
    rows = {g: {**{c: wmean(sub[c], sub['wt']) for c in cols}, 'N': len(sub)} for g, sub in df.groupby('group')}
    return pd.DataFrame(rows).T.loc[order]
d['group'] = np.select(
    [(d.vote20 == 'D') & (d.vote24 == 'D'), (d.vote20 == 'D') & (d.vote24 == 'R'),
     (d.vote20 == 'NV') & (d.vote24 == 'D'), (d.vote20 == 'NV') & (d.vote24 == 'R'),
     (d.vote20 == 'R') & (d.vote24 == 'R'), (d.vote20 == 'R') & (d.vote24 == 'D'), (d.vote20 == 'D') & (d.vote24 == 'NV')],
    ['Biden->Harris', 'Biden->Trump', 'New->Harris', 'New->Trump', 'Trump->Trump', 'Trump->Harris', 'Biden->stay home'], None)
ORDER = ['Biden->Harris', 'Biden->stay home', 'Biden->Trump', 'New->Harris', 'New->Trump', 'Trump->Harris', 'Trump->Trump']
def sc(x, lo, hi): return (x - lo) / (hi - lo)
# indices from nb 14
d['anti_elite'] = pd.concat([d.big_interests, sc(5 - d.govt_corrupt, 0, 4), sc(d.trust_govt, 1, 5), sc(d.outraged, 1, 5),
    sc(5 - d.officials_dont_care, 0, 4), sc(5 - d.no_say_in_govt, 0, 4), d.strong_leader_agree, sc(5 - d.cses_business_run, 0, 4), sc(5 - d.cses_referendums, 0, 4)], axis=1).mean(axis=1)
d['econ_left'] = pd.concat([sc(d.scale_services, 1, 7), sc(7 - d.scale_health_ins, 0, 6), sc(7 - d.scale_jobs_income, 0, 6),
    (d.min_wage == 1).astype(float).where(d.min_wage.notna()), sc(7 - d.paid_leave, 0, 6)], axis=1).mean(axis=1)
d['immig_cons'] = pd.concat([sc(7 - d.wall, 0, 6), sc(7 - d.dreamers, 0, 6), sc(7 - d.birthright_end, 0, 6),
    sc(d.immig_levels, 1, 5), sc(7 - d.immig_crime, 0, 6), sc(d.path_citizenship, 1, 7)], axis=1).mean(axis=1)
d['culture_cons'] = pd.concat([sc(d.trans_bathroom, 1, 7), sc(7 - d.trans_sports_ban, 0, 6), sc(d.trans_military, 1, 7),
    d.dei_campus.map({1: 0.0, 2: 1.0, 3: 0.5}), sc(d.diversity_better, 1, 5), sc(d.gay_marriage, 1, 7), sc(4 - d.abortion_std, 0, 3)], axis=1).mean(axis=1)
print(d.group.value_counts())

group
Biden->Harris       1945
Trump->Trump        1437
New->Trump           198
New->Harris          166
Biden->stay home     142
Biden->Trump          69
Trump->Harris         58
Name: count, dtype: int64


## 1. Anti-corporate or anti-politician?

Two families of anti-elite sentiment. **Anti-corporate:** big-business thermometer (cold = anti), "rich and powerful make it hard to get ahead" (describes view very/extremely well), millionaire tax (favor), campaign-spending limits (favor), labor-union thermometer (warm). **Anti-politician:** government run by a few big interests, most/all in government corrupt, officials don't care, no say, strong leader who bends rules, business leaders should run things.

In [2]:
d['rich_block_agree'] = (d.rich_powerful_block >= 4).astype(float).where(d.rich_powerful_block.notna())
d['millionaire_tax_favor'] = (d.millionaire_tax == 1).astype(float).where(d.millionaire_tax.notna())
d['spend_limits_favor'] = (d.campaign_spend_limits == 1).astype(float).where(d.campaign_spend_limits.notna())
d['corrupt_most_all'] = (d.govt_corrupt <= 2).astype(float).where(d.govt_corrupt.notna())
d['officials_dont_care_agree'] = (d.officials_dont_care <= 2).astype(float).where(d.officials_dont_care.notna())
d['no_say_agree'] = (d.no_say_in_govt <= 2).astype(float).where(d.no_say_in_govt.notna())
d['strong_leader_agree01'] = (d.cses_strong_leader <= 2).astype(float).where(d.cses_strong_leader.notna())
d['business_run_agree'] = (d.cses_business_run <= 2).astype(float).where(d.cses_business_run.notna())
d['trust_govt_never'] = (d.trust_govt == 5).astype(float).where(d.trust_govt.notna())
corp = ['therm_big_business', 'therm_unions', 'rich_block_agree', 'millionaire_tax_favor', 'spend_limits_favor']
pol = ['big_interests', 'corrupt_most_all', 'officials_dont_care_agree', 'no_say_agree', 'trust_govt_never', 'strong_leader_agree01', 'business_run_agree']
t = grp_table(d[d.group.isin(ORDER)], corp + pol, ORDER)
print('Anti-corporate items (thermometers 0-100; others share agreeing/favoring):')
print(t[corp + ['N']].round(2))
print('\nAnti-politician items (share):')
print(t[pol + ['N']].round(2))
t.to_csv(TABLES_DIR / 'anes_anticorp_vs_antipol_by_group.csv')
# composite: anti-corporate index and anti-politician index, 0-1
d['anti_corp'] = pd.concat([sc(100 - d.therm_big_business, 0, 100), sc(d.rich_powerful_block, 1, 5), d.millionaire_tax_favor, d.spend_limits_favor], axis=1).mean(axis=1)
d['anti_pol'] = pd.concat([d.big_interests, sc(5 - d.govt_corrupt, 0, 4), sc(5 - d.officials_dont_care, 0, 4), sc(5 - d.no_say_in_govt, 0, 4), sc(d.trust_govt, 1, 5)], axis=1).mean(axis=1)
d['authoritarian'] = pd.concat([d.strong_leader_agree, sc(5 - d.cses_business_run, 0, 4)], axis=1).mean(axis=1)
t2 = grp_table(d[d.group.isin(ORDER)], ['anti_corp', 'anti_pol', 'authoritarian'], ORDER)
print('\nIndices (0-1):'); print(t2.round(3))
bh, bt = d[d.group == 'Biden->Harris'], d[d.group == 'Biden->Trump']
print('\nSwitcher minus loyal-Democrat gap in pooled SDs:')
for c in ['anti_corp', 'anti_pol', 'authoritarian']:
    print(f'  {c:14s} {(wmean(bt[c], bt.wt) - wmean(bh[c], bh.wt)) / d[c].std():+.2f}')

Anti-corporate items (thermometers 0-100; others share agreeing/favoring):
                  therm_big_business  therm_unions  rich_block_agree  \
Biden->Harris                  38.41         70.61              0.49   
Biden->stay home               42.71         59.32              0.45   
Biden->Trump                   49.85         55.55              0.32   
New->Harris                    39.16         68.82              0.55   
New->Trump                     49.31         52.87              0.32   
Trump->Harris                  44.71         64.05              0.33   
Trump->Trump                   53.12         50.92              0.18   

                  millionaire_tax_favor  spend_limits_favor       N  
Biden->Harris                      0.88                0.79  1945.0  
Biden->stay home                   0.67                0.54   142.0  
Biden->Trump                       0.70                0.52    69.0  
New->Harris                        0.80                0.64   166.0 

## 2. Does anti-corporate sentiment predict leaving?

The capture hypothesis makes a directional prediction: among Biden 2020 voters, those most hostile to big business and most convinced the rich block everyone else should have been the *most* likely to leave — they judged the party captured. The rival reading is that anti-corporate Biden voters saw Republicans as the more corporate party and stayed, while the leavers were anti-politician and authoritarian-leaning but not especially anti-corporate.

In [3]:
b = d[(d.vote20 == 'D') & d.vote24.isin(['D', 'R'])].copy()
b['switch'] = (b.vote24 == 'R').astype(float)
def fit(cols, label, std=True):
    sub = b.dropna(subset=cols + ['switch'])
    Z = sub[cols].copy()
    if std:
        for c in cols: Z[c] = (Z[c] - Z[c].mean()) / Z[c].std()
    r = sm.GLM(sub['switch'], sm.add_constant(Z), family=sm.families.Binomial(), freq_weights=sub['wt'] / sub['wt'].mean()).fit(cov_type='HC1')
    out = pd.DataFrame({'coef': r.params, 'se': r.bse, 'p': r.pvalues}).round(3)
    print(f'\n{label}: N={len(sub)}, events={int(sub.switch.sum())}'); print(out); return out
m1 = fit(['anti_corp', 'anti_pol', 'authoritarian'], 'M1 three anti-elite families')
m2 = fit(['anti_corp', 'anti_pol', 'authoritarian', 'econ_left', 'immig_cons', 'culture_cons'], 'M2 + positions')
# item by item, each alone with the position controls
rows = []
for c in corp + pol:
    sub = b.dropna(subset=[c, 'econ_left', 'immig_cons', 'culture_cons'])
    Z = sub[[c, 'econ_left', 'immig_cons', 'culture_cons']].copy()
    for k in ['econ_left', 'immig_cons', 'culture_cons']: Z[k] = (Z[k] - Z[k].mean()) / Z[k].std()
    if c.startswith('therm'): Z[c] = Z[c] / 100
    r = sm.GLM(sub['switch'], sm.add_constant(Z), family=sm.families.Binomial(), freq_weights=sub['wt'] / sub['wt'].mean()).fit(cov_type='HC1')
    rows.append({'item': c, 'coef': r.params[c], 'se': r.bse[c], 'p': r.pvalues[c], 'N': len(sub), 'events': int(sub.switch.sum()),
                 'loyal_mean': wmean(bh[c], bh.wt), 'switcher_mean': wmean(bt[c], bt.wt)})
items = pd.DataFrame(rows).set_index('item').round(3)
print('\nEach item alone, net of economic/immigration/cultural indices (thermometers per 100 points; binaries as 0/1):')
print(items)
items.to_csv(TABLES_DIR / 'anes_antielite_items_switch.csv')
pd.concat({'M1': m1, 'M2': m2}).to_csv(TABLES_DIR / 'anes_anticorp_switch_models.csv')


M1 three anti-elite families: N=1937, events=62
                coef     se      p
const         -4.059  0.167  0.000
anti_corp     -0.372  0.119  0.002
anti_pol       0.631  0.144  0.000
authoritarian  0.826  0.117  0.000

M2 + positions: N=1937, events=62
                coef     se      p
const         -4.772  0.229  0.000
anti_corp     -0.179  0.136  0.188
anti_pol       0.527  0.152  0.001
authoritarian  0.632  0.126  0.000
econ_left     -0.255  0.139  0.068
immig_cons     0.387  0.148  0.009
culture_cons   0.843  0.177  0.000

Each item alone, net of economic/immigration/cultural indices (thermometers per 100 points; binaries as 0/1):
                            coef     se      p     N  events  loyal_mean  \
item                                                                       
therm_big_business         1.198  0.520  0.021  1922      62      38.411   
therm_unions              -1.270  0.662  0.055  1924      62      70.606   
rich_block_agree          -0.222  0.290  0.443

In [4]:
# Simple partition: switch rate by anti-corporate tercile x anti-politician tercile among Biden-2020 voters
for c in ['anti_corp', 'anti_pol']:
    b[c + '_t'] = pd.qcut(b[c], 3, labels=['low', 'mid', 'high'])
part = b.dropna(subset=['anti_corp_t', 'anti_pol_t']).groupby(['anti_corp_t', 'anti_pol_t']).apply(
    lambda s: pd.Series({'switch_rate': wmean(s.switch, s.wt), 'N': len(s), 'switchers': int(s.switch.sum())}))
print('Switch rate among Biden-2020 voters, anti-corporate tercile x anti-politician tercile:')
print(part.round(3).unstack('anti_pol_t'))
part.to_csv(TABLES_DIR / 'anes_switch_by_anticorp_x_antipol.csv')
for c in ['anti_corp', 'anti_pol']:
    print(f'\n{c} tercile alone:')
    print(b.groupby(c + '_t').apply(lambda s: pd.Series({'switch_rate': wmean(s.switch, s.wt), 'N': len(s), 'switchers': int(s.switch.sum())})).round(3))

Switch rate among Biden-2020 voters, anti-corporate tercile x anti-politician tercile:
            switch_rate                    N               switchers        \
anti_pol_t          low    mid   high    low    mid   high       low   mid   
anti_corp_t                                                                  
low               0.030  0.086  0.149  295.0  217.0  182.0       9.0  14.0   
mid               0.006  0.059  0.028  298.0  224.0  193.0       6.0   5.0   
high              0.000  0.006  0.030  145.0  178.0  277.0       0.0   1.0   

                   
anti_pol_t   high  
anti_corp_t        
low          22.0  
mid           6.0  
high          6.0  

anti_corp tercile alone:
             switch_rate      N  switchers
anti_corp_t                               
low                0.080  694.0       45.0
mid                0.029  715.0       17.0
high               0.015  600.0        7.0

anti_pol tercile alone:
            switch_rate      N  switchers
anti_pol_t      

## 3. What they thought of the Democratic Party

Pre-election party thermometers (asked before the vote), CSES 0-10 like/dislike (post), and left-right placement of the parties and self (post, 0 left – 10 right). Two readings of "lost trust": the leavers saw the Democrats as having moved away from them (placement distance grew), or as indistinguishable from the Republicans (the parties converge in their eyes).

In [5]:
d['therm_party_gap'] = d.therm_rep_party - d.therm_dem_party
d['like_party_gap'] = d.like_rep_party - d.like_dem_party
d['lr_dist_dem'] = (d.lr_self - d.lr_dem_party).abs()
d['lr_dist_rep'] = (d.lr_self - d.lr_rep_party).abs()
d['lr_party_spread'] = d.lr_rep_party - d.lr_dem_party
d['dem_left_of_self'] = d.lr_self - d.lr_dem_party   # positive = sees Dems to their left
pc = ['therm_dem_party', 'therm_rep_party', 'like_dem_party', 'like_rep_party', 'lr_self', 'lr_dem_party', 'lr_rep_party', 'lr_dist_dem', 'lr_dist_rep', 'lr_party_spread', 'dem_left_of_self']
t = grp_table(d[d.group.isin(ORDER)], pc, ORDER)
print(t.round(2))
t.to_csv(TABLES_DIR / 'anes_party_perception_by_group.csv')
# distribution of Dem-party thermometer among switchers vs loyal
for g in ['Biden->Harris', 'Biden->stay home', 'Biden->Trump']:
    s = d[d.group == g]
    print(f'\n{g}: Dem party therm <=30: {100*wshare(s.therm_dem_party <= 30, s.wt):.0f}%, 31-60: {100*wshare(s.therm_dem_party.between(31, 60), s.wt):.0f}%, >60: {100*wshare(s.therm_dem_party > 60, s.wt):.0f}%  (pre-election)')
# Which is the better description of the switcher: Dems too far left, or no difference between parties?
sw = d[d.group == 'Biden->Trump']
print('\nSwitchers: sees Democratic Party to the left of self (dem_left_of_self > 0): %.0f%%; at or right of self: %.0f%%' % (
    100*wshare(sw.dem_left_of_self > 0, sw.wt), 100*wshare(sw.dem_left_of_self <= 0, sw.wt)))
print('Switchers: closer to Rep party than Dem party on left-right: %.0f%%; equidistant: %.0f%%; closer to Dems: %.0f%%' % (
    100*wshare(sw.lr_dist_rep < sw.lr_dist_dem, sw.wt), 100*wshare(sw.lr_dist_rep == sw.lr_dist_dem, sw.wt), 100*wshare(sw.lr_dist_rep > sw.lr_dist_dem, sw.wt)))

                  therm_dem_party  therm_rep_party  like_dem_party  \
Biden->Harris               73.18            17.87            7.55   
Biden->stay home            62.27            29.96            6.30   
Biden->Trump                36.39            53.99            3.27   
New->Harris                 63.96            25.45            7.25   
New->Trump                  22.17            72.48            2.98   
Trump->Harris               51.94            49.18            5.99   
Trump->Trump                14.81            75.97            1.79   

                  like_rep_party  lr_self  lr_dem_party  lr_rep_party  \
Biden->Harris               1.86     3.83          3.63          7.75   
Biden->stay home            3.50     5.55          4.49          5.39   
Biden->Trump                6.58     5.87          3.57          6.82   
New->Harris                 2.54     4.27          3.64          7.30   
New->Trump                  7.47     6.99          2.50          7.64   
T

## 4. Lost faith or transferred faith?

"Which party would better handle" your own most important problem, and each named issue. A voter who lost faith in Democrats without gaining it in Republicans says "no difference"; a voter who transferred faith says "Republicans." The stay-home group is the cleanest test of lost-without-transferred.

In [6]:
rows = []
for g in ORDER:
    s = d[(d.group == g) & d.mip1_party.notna()]
    rows.append({'group': g, 'Democrats': wshare(s.mip1_party == 1, s.wt), 'Republicans': wshare(s.mip1_party == 2, s.wt), 'no difference': wshare(s.mip1_party == 3, s.wt), 'N': len(s)})
t = pd.DataFrame(rows).set_index('group')
print('Party better on own most important problem (row %):'); print((100*t[['Democrats', 'Republicans', 'no difference']]).round(1).assign(N=t.N))
# "no difference" count across the eight ownership items
own = [c for c in d.columns if c.startswith('own_') and c not in ('own_democracy', 'own_racial_ineq')]
d['own_nodiff_share'] = (d[own] == 3).sum(axis=1) / d[own].notna().sum(axis=1)
d['own_rep_share'] = (d[own] == 2).sum(axis=1) / d[own].notna().sum(axis=1)
d['own_dem_share'] = (d[own] == 1).sum(axis=1) / d[own].notna().sum(axis=1)
t2 = grp_table(d[d.group.isin(ORDER)], ['own_dem_share', 'own_rep_share', 'own_nodiff_share'], ORDER)
print('\nMean share of eight issues on which the respondent says Democrats / Republicans / no difference:'); print(t2.round(3))
t2.to_csv(TABLES_DIR / 'anes_ownership_shares_by_group.csv')

Party better on own most important problem (row %):
                  Democrats  Republicans  no difference     N
group                                                        
Biden->Harris          69.3          4.1           26.7  1922
Biden->stay home       35.7         16.1           48.3   141
Biden->Trump            3.0         69.7           27.4    68
New->Harris            54.9          5.6           39.5   162
New->Trump              2.6         83.6           13.8   197
Trump->Harris          38.8         27.7           33.5    57
Trump->Trump            1.1         90.5            8.4  1411

Mean share of eight issues on which the respondent says Democrats / Republicans / no difference:
                  own_dem_share  own_rep_share  own_nodiff_share       N
Biden->Harris             0.671          0.079             0.250  1945.0
Biden->stay home          0.391          0.202             0.407   142.0
Biden->Trump              0.169          0.598             0.232    69.0


## 5. Who was reachable: the "moderate" question

Self-placement of each group on the 0-10 left-right scale, and issue positions by self-placement among the leavers. If the leavers call themselves moderate but want government health insurance, a higher minimum wage, and a millionaire tax, "moderate" is a self-label, not a policy centrism the party could court with centrist policy.

In [7]:
d['lr_bin'] = pd.cut(d.lr_self, [-0.1, 3, 6, 10], labels=['left (0-3)', 'center (4-6)', 'right (7-10)'])
for g in ['Biden->Harris', 'Biden->Trump', 'New->Trump', 'Trump->Trump']:
    s = d[(d.group == g) & d.lr_bin.notna()]
    print(f'{g:14s} self-placement: ' + ', '.join(f'{k} {100*wshare(s.lr_bin == k, s.wt):.0f}%' for k in ['left (0-3)', 'center (4-6)', 'right (7-10)']))
d['gov_health_ins'] = (d.scale_health_ins <= 3).astype(float).where(d.scale_health_ins.notna())
d['raise_min_wage'] = (d.min_wage == 1).astype(float).where(d.min_wage.notna())
pos = ['gov_health_ins', 'raise_min_wage', 'millionaire_tax_favor', 'rich_block_agree', 'spend_limits_favor', 'therm_big_business', 'immig_cons', 'culture_cons']
lv = d[d.group.isin(['Biden->Trump', 'New->Trump']) & d.lr_bin.notna()]
t = lv.groupby('lr_bin').apply(lambda s: pd.Series({**{c: wmean(s[c], s.wt) for c in pos}, 'N': len(s)}))
print('\nLeavers + new Trump voters, positions by self-placement:'); print(t.round(2))
t.to_csv(TABLES_DIR / 'anes_leavers_positions_by_selfplacement.csv')
# the comparison group: self-described centrists who stayed with Harris
cen = d[(d.group == 'Biden->Harris') & (d.lr_bin == 'center (4-6)')]
print('\nBiden->Harris centrists (N=%d): ' % len(cen) + ', '.join(f'{c} {wmean(cen[c], cen.wt):.2f}' for c in pos))

Biden->Harris  self-placement: left (0-3) 46%, center (4-6) 43%, right (7-10) 11%
Biden->Trump   self-placement: left (0-3) 13%, center (4-6) 48%, right (7-10) 39%
New->Trump     self-placement: left (0-3) 4%, center (4-6) 38%, right (7-10) 58%
Trump->Trump   self-placement: left (0-3) 2%, center (4-6) 22%, right (7-10) 76%

Leavers + new Trump voters, positions by self-placement:
              gov_health_ins  raise_min_wage  millionaire_tax_favor  \
lr_bin                                                                
left (0-3)              0.35            0.29                   0.39   
center (4-6)            0.38            0.50                   0.63   
right (7-10)            0.41            0.43                   0.57   

              rich_block_agree  spend_limits_favor  therm_big_business  \
lr_bin                                                                   
left (0-3)                0.18                0.32               62.44   
center (4-6)              0.44        

## 6. Summary

In [8]:
summary = pd.DataFrame({
    'big business therm': grp_table(d[d.group.isin(ORDER)], ['therm_big_business'], ORDER)['therm_big_business'],
    'rich block (%)': 100*grp_table(d[d.group.isin(ORDER)], ['rich_block_agree'], ORDER)['rich_block_agree'],
    'millionaire tax (%)': 100*grp_table(d[d.group.isin(ORDER)], ['millionaire_tax_favor'], ORDER)['millionaire_tax_favor'],
    'anti_corp': grp_table(d[d.group.isin(ORDER)], ['anti_corp'], ORDER)['anti_corp'],
    'anti_pol': grp_table(d[d.group.isin(ORDER)], ['anti_pol'], ORDER)['anti_pol'],
    'authoritarian': grp_table(d[d.group.isin(ORDER)], ['authoritarian'], ORDER)['authoritarian'],
    'Dem party therm (pre)': grp_table(d[d.group.isin(ORDER)], ['therm_dem_party'], ORDER)['therm_dem_party'],
    'sees Dems left of self': grp_table(d[d.group.isin(ORDER)], ['dem_left_of_self'], ORDER)['dem_left_of_self'],
    'no-diff share (8 issues)': grp_table(d[d.group.isin(ORDER)], ['own_nodiff_share'], ORDER)['own_nodiff_share'],
    'N': grp_table(d[d.group.isin(ORDER)], ['anti_corp'], ORDER)['N']})
print(summary.round(2))
summary.to_csv(TABLES_DIR / 'anes_capture_summary_by_group.csv')
print('\nSaved 9 tables (anes_*.csv)')

                  big business therm  rich block (%)  millionaire tax (%)  \
Biden->Harris                  38.41           49.44                87.51   
Biden->stay home               42.71           45.26                66.80   
Biden->Trump                   49.85           32.49                69.63   
New->Harris                    39.16           54.61                79.68   
New->Trump                     49.31           31.81                55.29   
Trump->Harris                  44.71           33.02                71.34   
Trump->Trump                   53.12           17.52                37.99   

                  anti_corp  anti_pol  authoritarian  Dem party therm (pre)  \
Biden->Harris          0.73      0.61           0.18                  73.18   
Biden->stay home       0.60      0.71           0.34                  62.27   
Biden->Trump           0.54      0.71           0.49                  36.39   
New->Harris            0.68      0.67           0.24               

## Limitations

- **69 switchers, 142 stay-homes.** Same caveat as notebook 14; the item-by-item models in §2 have 60-odd events across 12 items and will produce some spurious significance by chance.
- **Post-election measurement** for everything except the party thermometers and the pre-election anti-politician items (big interests, corrupt, trust, outraged). The CSES party evaluations and left-right placements are post; a Trump voter rates the Democratic Party after voting against it.
- **"Anti-corporate" is four items;** the ANES has no antitrust, price-gouging, or "Democrats are beholden to donors" item. The big-business thermometer and the "rich and powerful" item are the closest available. A party-specific capture perception — "the Democrats serve corporate interests" — is not asked by any survey on disk; the test here is whether general anti-corporate sentiment predicts leaving the Democrats, which is a necessary but not sufficient condition for the hypothesis.
- **Left-right placement of parties** is a CSES 0-10 scale that Americans use loosely; the distance measures are ordinal at best.